# 07 - Live Evaluation Service: MAP, nDCG, Precision@10, Recall

هذا أهم Notebook للمقابلة.

يجب أن يظهر في الخرج:
- عدد الـ queries المستخدمة في التقييم.
- عدد qrels المستخدمة.
- توزيع relevance.
- أن التقييم يحسب query by query.
- جدول Summary للموديلات.
- Charts خصوصاً MAP و nDCG.

رابط الداتاسيت الرسمي:
https://ir-datasets.com/medline.html#medline/2004/trec-genomics-2005

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
from IPython.display import display, Markdown
from src.datasets.loader import load_queries, load_qrels

DATASET_ID = "medline/2004/trec-genomics-2005"
DATASET_URL = "https://ir-datasets.com/medline.html#medline/2004/trec-genomics-2005"

display(Markdown(f"**Dataset page / qrels verification:** [{DATASET_ID}]({DATASET_URL})"))

queries_df = load_queries("main")
qrels_df = load_qrels("main")

print("FULL queries count from dataset:", len(queries_df))
print("FULL qrels count from dataset:", len(qrels_df))
print("Unique query_id in qrels:", qrels_df["query_id"].astype(str).nunique())
print("Unique doc_id in qrels:", qrels_df["doc_id"].astype(str).nunique())
print("Relevance distribution:")
display(qrels_df["relevance"].value_counts().sort_index().rename_axis("relevance").reset_index(name="count"))

In [ ]:
# أثناء التجريب يمكنك وضع LIMIT_QUERIES = 5 لتسريع notebook.
# في المقابلة أو التقرير النهائي ضعها None ليظهر العدد الكامل 50.
LIMIT_QUERIES = None

if LIMIT_QUERIES is not None:
    queries_eval = queries_df.head(LIMIT_QUERIES).copy()
    allowed_qids = set(queries_eval["query_id"].astype(str))
    qrels_eval = qrels_df[qrels_df["query_id"].astype(str).isin(allowed_qids)].copy()
else:
    queries_eval = queries_df.copy()
    qrels_eval = qrels_df.copy()

print("Queries used in THIS evaluation run:", len(queries_eval))
print("Qrels used in THIS evaluation run:", len(qrels_eval))
print("Query IDs used:", list(queries_eval["query_id"].astype(str))[:20], "...")

In [ ]:
from src.evaluation import EvaluationConfig, EvaluationService
from src.retrieval import BM25Retriever, TfidfRetriever, BertRetriever, Word2VecRetriever, ParallelHybridRetriever, SerialHybridRetriever

retrievers = {}
RUN_DEPTH = 100

if TERRIER_INDEX_PATH.exists() and DB_PATH.exists():
    retrievers["bm25"] = BM25Retriever(index_path=str(TERRIER_INDEX_PATH), db_path=str(DB_PATH), top_k=RUN_DEPTH)
    retrievers["tfidf"] = TfidfRetriever(index_path=str(TERRIER_INDEX_PATH), db_path=str(DB_PATH), top_k=RUN_DEPTH)
else:
    print("Skipping BM25/TF-IDF because Terrier index or DB is missing.")

if (BERT_INDEX_DIR / "index.faiss").exists() and DB_PATH.exists():
    retrievers["bert"] = BertRetriever(index_dir=str(BERT_INDEX_DIR), db_path=str(DB_PATH), top_k=RUN_DEPTH)
else:
    print("Skipping BERT because BERT FAISS index is missing.")

if (WORD2VEC_INDEX_DIR / "index.faiss").exists() and (WORD2VEC_INDEX_DIR / "word2vec.model").exists() and DB_PATH.exists():
    retrievers["word2vec"] = Word2VecRetriever(index_dir=str(WORD2VEC_INDEX_DIR), db_path=str(DB_PATH), top_k=RUN_DEPTH)
else:
    print("Skipping Word2Vec because artifacts are missing.")

# Hybrid models only if required artifacts exist.
if "bm25" in retrievers:
    try:
        retrievers["serial_bm25_bert"] = SerialHybridRetriever(
            first_stage="bm25",
            second_stage="bert",
            index_path=str(TERRIER_INDEX_PATH),
            db_path=str(DB_PATH),
            word2vec_index_dir=str(WORD2VEC_INDEX_DIR),
            candidate_k=100,
            top_k=RUN_DEPTH,
        )
    except Exception as exc:
        print("Skipping serial hybrid:", type(exc).__name__, exc)

parallel_models = []
if "bm25" in retrievers: parallel_models.append("bm25")
if "tfidf" in retrievers: parallel_models.append("tfidf")
if "word2vec" in retrievers: parallel_models.append("word2vec")
if "bert" in retrievers: parallel_models.append("bert")

if len(parallel_models) >= 2:
    try:
        retrievers["parallel_hybrid_rrf"] = ParallelHybridRetriever(
            models=parallel_models,
            index_path=str(TERRIER_INDEX_PATH),
            db_path=str(DB_PATH),
            word2vec_index_dir=str(WORD2VEC_INDEX_DIR),
            bert_index_dir=str(BERT_INDEX_DIR),
            per_model_k=100,
            top_k=RUN_DEPTH,
            rrf_k=60,
        )
    except Exception as exc:
        print("Skipping parallel hybrid:", type(exc).__name__, exc)

print("Retrievers ready for evaluation:", list(retrievers.keys()))
assert retrievers, "No retriever is available. Build the indexes first."

In [ ]:
config = EvaluationConfig(
    run_depth=RUN_DEPTH,
    precision_k=10,
    recall_k=RUN_DEPTH,
    ndcg_k=10,
    map_depth=RUN_DEPTH,
    min_relevance=1,
    continue_on_error=True,
    save_charts=True,
    save_excel=True,
)

service = EvaluationService(
    output_dir=REPORTS_DIR,
    config=config,
)

print("Starting evaluation now...")
print("Queries used:", len(queries_eval))
print("Qrels used:", len(qrels_eval))
print("Models:", list(retrievers.keys()))

result = service.evaluate_models(
    retrievers=retrievers,
    queries_df=queries_eval,
    qrels_df=qrels_eval,
)

print("Evaluation finished.")
print("Summary CSV:", result["summary_csv"])
print("Summary XLSX:", result["summary_xlsx"])
print("Charts:", result["charts"])

In [ ]:
summary = pd.DataFrame(result["summary_rows"])
display(summary)

In [ ]:
import matplotlib.pyplot as plt

metric_candidates = ["map", "ndcg_at_10", "precision_at_10", f"recall_at_{RUN_DEPTH}"]
metric_columns = [c for c in metric_candidates if c in summary.columns]
print("Metric columns plotted:", metric_columns)

for metric in metric_columns:
    ax = summary.plot(kind="bar", x="model", y=metric, legend=False, figsize=(8,4), title=f"{metric} comparison")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()